### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("..")

# Imports
from typing import Any
from openai import OpenAI
import random
from pathlib import Path
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
import json

import threading

warnings.filterwarnings("ignore")

In [ ]:
# Configuration
STEP_X = 3  # Variable int - go until this many patient turns in the conversation
N_SAMPLES_PER_DIAGNOSIS = 10
BREAK_AFTER_ONE = True  # Set to True to break after first sample for inspection

from dotenv import load_dotenv
load_dotenv("../../.env")

openai_client = OpenAI()

#### Run an AI eval
- Measure consistency between questions; answers; also measure "missing"

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class InterAnswerConsistencyEvaluation(BaseModel):
    rating: Literal["consistent", "not_consistent"] = Field(description="The rating of the answer consistency, either 'consistent' or 'not_consistent'")
    reasoning: str = Field(description="The reasoning for the rating")

class GTAnswerConsistencyEvaluation(BaseModel):
    rating: Literal["consistent", "not_consistent", "na"] = Field(description="The rating of the answer consistency, either 'consistent' or 'not_consistent' or 'na' if not applicable")
    reasoning: str = Field(description="The reasoning for the rating")


class TwoAnswerRating(BaseModel):
    a: GTAnswerConsistencyEvaluation = Field(description="The evaluation of answer A")
    b: GTAnswerConsistencyEvaluation = Field(description="The evaluation of answer B")

INTER_ANSWER_CONSISTENCY_EVALUATION_PROMPT = """You are evaluating answers from a single simulated patient during medical history taking.

For each case, you will receive:
    * Conversation so far between doctor and patient (context up to the moment of the question).
    * Question A and Answer A.
    * Question B and Answer B.

Both answers come from the same patient simulation and are responses to slightly different variants of the doctor’s question. Your task is to decide whether A and B are `consistent` or `not_consistent` with each other.

# Core task

Treat this as checking whether a single real patient could reasonably have given both Answer A and Answer B in this situation. You are not judging medical quality, only whether the two answers could both be true for the same patient at (roughly) the same time, given the conversation so far.

# Definition of labels
* `consistent`
    Label as consistent if:
        - A and B can both be true for the same patient at the same time, and
        - There is no clear logical or factual conflict between them, even when considering the conversation so far.

    Situations that are still consistent:
        - One answer is more detailed, the other is shorter or incomplete.
        - One answer gives a subset of the information in the other.
        - One answer adds new symptoms, diagnoses, or history without contradicting anything in the other answer or the conversation so far.
        - One answer is generic and the other is specific, but both can fit into a single coherent patient story.
        - One answer is like: “There’s nothing else relevant to mention.” and the other is: “I don’t smoke and I did not travel abroad recently.” This is consistent, because the first answer does not explicitly deny the facts in the second one.

    Minor differences in vague descriptions (e.g., “a few days” vs. “about a week”) should not be treated as contradictions unless they are clearly incompatible in context.

* `not_consistent`
    Label as not_consistent if:
        - A and B cannot both be true for the same patient at the same time, given the conversation so far, or  
        - There is a clear contradiction about clinically relevant facts.

    Typical examples of not_consistent:
        - One answer says a symptom is present, the other says it is not present (for the same time frame and context).
        - One answer states a diagnosis, event, or past illness did happen, the other explicitly says it never happened.
        - One answer says the patient takes a specific medication regularly, the other says the patient does not take that medication (or has never taken it), when both clearly talk about the same kind of use (e.g., long-term home medication).
        - One answer states there is a relevant allergy or repeated allergic reaction, the other explicitly says there are no allergies, and both clearly refer to the same type of allergy information.

    Do not over-interpret small differences in wording. Focus on clear, clinically relevant conflicts.

# Interpreting time frames and scope

Think like a doctor and pay attention to what time frame and which aspect of the history a question is targeting.

1. Current vs. past symptoms
Questions phrased in the present tense like
- “Do you have any chest pain?”
- “Are there any symptoms related to the nervous system, such as problems with movement or sensation?”
usually refer to the current episode or current baseline, not to the entire lifetime history.
A denial of current symptoms can still be consistent with a past history of the same problem (e.g., an old stroke, remote seizures, prior gait disorder), as long as the answer does not explicitly deny the past history.
However, if the context or conversation clearly describes that a symptom is currently present or worsening (e.g., current somnolence, current gait disturbance) and an answer clearly states that there are no such symptoms, this should be labeled not_consistent.
In other words:
Only treat it as a contradiction if A and B are talking about the same symptom in the same time frame (e.g., “right now” or “in this recent episode”).

2. Regular medication vs. acute symptom treatment
Distinguish between:
- Long-term / home medication: drugs taken regularly (e.g., daily antihypertensives, anticoagulants, insulin).
- Short-term / situational medication: drugs taken once or briefly for the current complaint (e.g., one dose of ibuprofen for today’s headache).

When you see questions like:
“Do you take any medication?”
“What medications do you take regularly?”
they usually refer to ongoing, regular medication, not to one-time self-medication for the current symptoms.

Therefore:
Answers such as
A: “I don’t take any medication.”
B: “I took one dose of pain medication today for this headache.”
can be consistent, because they refer to different types of medication use (chronic vs. acute).

Only label `not_consistent` when the answers clearly talk about the same scope and time frame (e.g., both talk about long-term/home meds) and directly contradict each other (e.g., “I take warfarin every day” vs. “I don’t take any blood thinners at all.”).

3. Allergies and adverse reactions
Make sure to consider the context of the conversation and the question that was asked.

When comparing A and B:

If one answer clearly states there is no known allergy (e.g., “I have no known medication or food allergies.”) and the other answer clearly reports a relevant allergy or recurring allergic reaction, treat this as not_consistent. Both cannot be true at the same time for the same patient.

If one answer mentions only a mild, non-allergic side effect or something obviously not framed as an “allergy”, this does not automatically conflict with “no known allergies” and can still be consistent.
Focus on whether the two answers give incompatible statements about the presence or absence of clinically relevant allergy-type problems, not on fine details of classification.

# Use of the conversation context

- Use the conversation so far to interpret meaning, timing, and scope, but keep the focus on whether Answer A and Answer B conflict with each other.
- Differences in level of detail are not contradictions by themselves.
- If something was mentioned earlier in the conversation but is not repeated in one of the answers, that alone does not make the answers inconsistent. Only treat it as not_consistent if:
- A and B directly disagree on a fact that matters clinically,
- and they clearly refer to the same time frame and same concept (same symptom, same medication use type, same allergy information, etc.).

CAVE: Sometimes answer can differ, especially if they contain no medical information about the patient, such as situations where the doctor explains the next steps and the patient confirms, or asks questions themselves. In these cases, you should output `consistent`, if the 2 answers do not clearly conflict with the patient's medical history.

If A and B can both fit into a single plausible patient story, given the context and these rules, output consistent.
If A and B cannot both be true for the same patient in that shared context, output not_consistent.
""".strip()



ANSWER_GT_CONSISTENCY_EVALUATION_PROMPT = """
You are given a **ground_truth** history of present illness (HPI) for a single patient, a **Conversation so far** between a doctor and this patient, and two alternative question–answer pairs (A and B) that continue this same conversation:

- Question A and Answer A
- Question B and Answer B

Your task: For each answer independently, decide if it is **consistent**, **not_consistent**, or **na** with the **ground_truth**, given the conversation context and the corresponding question.

Important:
- Treat **ground_truth as the authoritative source about the patient**, but remember that parts of the ground_truth may reflect information that comes from physical examination, tests, or events later in the hospital stay, which the patient cannot know or does not directly experience.
- You must **not** assume that the patient has access to internal notes, imaging results, lab values, neurological exam findings, or observations like “the patient appears somnolent” unless they are clearly described as something the patient reports themselves.
- Use the **Conversation so far** to understand what the patient already said; Question A/Question B define the time frame and focus of Answer A/Answer B.

You will see the input in this structured format:

<Ground Truth>
... ground_truth HPI text ...
</Ground Truth>

<Conversation so far>
... prior dialog between doctor and patient ...
</Conversation so far>
==========  
Question A:
... original_question ...

Answer A:
... original_answer ...
==========
Question B:
... question_variant ...

Answer B:
... new_answer ...

Both Answer A and Answer B are **alternative continuations** of the same conversation so far. They are **not** sequential. For the output you must:

1. Compare **Answer A** only against the ground_truth (plus Conversation so far and Question A).
2. Compare **Answer B** only against the ground_truth (plus Conversation so far and Question B).
3. Do **not** compare Answer A and Answer B with each other.

Your job:  
Return **two ratings**, one for Answer A and one for Answer B, each with an explanation.

# Definitions and rules

You must interpret the situation like a clinician who understands history taking, time frames, and typical question intent.

1. General consistency rules

- `consistent`:
    - The answer does **not contradict** the ground_truth, given the question and the conversation context.
    - If a disease, symptom, or other medical fact is **not mentioned** in the ground_truth and the answer indicates it is **absent** or “not known,” this is `consistent`.
    - If the answer describes a **normal state** (e.g. “I can exercise normally”) and the ground_truth does **not explicitly state a limitation**, this is `consistent`.
    - Differences in wording, level of detail, or completeness are allowed as long as there is **no explicit conflict** with the ground_truth.

- `not_consistent`:
    - The answer **contradicts something explicitly stated** in the ground_truth **for the same time frame and from the patient’s perspective**.
    - Typical explicit contradictions:
    - Symptom is clearly present **at the relevant time** in the ground_truth and is clearly denied in the answer.
    - Disease is clearly present in the ground_truth (e.g. a known chronic diagnosis) and is denied in the answer.
    - Ground_truth describes a functional limitation at the relevant time, but the answer claims normal function at that same time.

- `na`:
    - Use `na` only when the answer is **purely non-informative** and does not add any medical content, such as:
    - “Thank you.”
    - “I understand. What will happen now?”
    - The patient asks a question back without stating anything about symptoms, diseases, medications, history, lifestyle, etc.
    - If the answer contains **any medical / patient-related information** (symptoms, diseases, medication, dosages, family history, lifestyle, allergies, etc.), then you must use `consistent` or `not_consistent`, never `na` and ignore the non-relevant parts.

2. Time frame and question interpretation

You must carefully interpret the **time frame implied by the question**:

- If the question clearly refers to the **current situation** (e.g. “right now,” “at the moment,” “with this episode,” “today,” “since this started”), then:
    - Compare the answer primarily to **current or recent symptoms** in the ground_truth.
    - Past or historical problems in the ground_truth (e.g. “in 2019 the patient had a TIA,” “remote history of gait problems”) do **not** automatically contradict a **current** denial in the answer, as long as the ground_truth does not say the symptom is still present now.
    - Example: If the ground_truth mentions previous x years ago, but the question is “Do you currently have any problems with x?” and the answer is “No,” this can be `consistent` if the ground_truth does not clearly say that x problems are currently present.

- If the question refers to a **general or past time window** (e.g. “Have you ever…?”, “In the past…?”, “Before this episode…?”), then:
    - Compare to the full past history in the ground_truth.
    - If the ground_truth clearly documents a past event (e.g. prior stroke, prior seizures, prior myocardial infarction) and the patient denies **ever** having such a problem, this is `not_consistent`.

- When an answer says “I did not have such symptoms in the past,” interpret “in the past” as a broad time frame before the current episode, not necessarily “earlier this morning.” Compare it accordingly with the ground_truth.

3. What parts of the ground_truth the patient can reasonably know

The ground_truth may contain information from:
- The patient’s own reported history.
- Physical examinations.
- Imaging results.
- Lab results.
- Observations during an ER or hospital stay.

For consistency judgements, follow these rules:

- Assume the patient **knows** and can report:
    - Their long-standing diagnoses that are clearly documented (e.g. diabetes, hypertension, previous myocardial infarction, epilepsy), unless the text explicitly suggests these were unknown to the patient.
    - Their own **subjective symptoms and experiences** (e.g. chest pain, dyspnea, dizziness, headaches, falls they remember, functional limitations they perceive).
- Do **not** require the patient to match:
    - Purely clinician-observed states (e.g. “patient appears somnolent,” “on examination the patient has frontal gait disturbance”) if these are not described as something the patient experiences or recognizes.
    - Test or imaging findings (e.g. “CT shows infarct,” “MRI shows lesion,” “labs show anemia”) unless the ground_truth clearly states that the patient has been informed and is aware of the diagnosis in a way that would normally be part of their medical history (“known history of stroke,” “known tumor diagnosis,” etc.) or if this information is from earlier before the current conversation (yesterday, earlier this morning, etc.)

Therefore:
- If the ground_truth shows abnormal findings x only from an exam, the patient’s answer to a question like “Do you notice any problems with x?” can still be `consistent` if they say “no,” as long as the ground_truth does not clearly state that the patient perceives these problems.
- If the ground_truth clearly states that the patient regularly experiences a symptom (e.g. “patient reports daily x”) and the question asks about this same type of symptom in an appropriate time frame (current or general), then a denial in the answer is `not_consistent`.

4. Medication questions: chronic vs acute use

When interpreting questions about medications, you must distinguish between:
- **Chronic / regular medication use** (e.g. daily antihypertensives, long-term antidepressants).
- **Acute or short-term medication use** for the current complaint (e.g. “I took ibuprofen for this headache”).

Apply these rules:
- If the question is broad and typical for medication history, like:
    - “Do you take any medication?”
    - “What medicines do you take regularly?”
    - “Are you on any long-term treatment?”
    then you should interpret this as asking about **regular or long-term medication**.

    Examples:
    - If the ground_truth only mentions that the patient took a **single dose of analgesic for the current symptom**, and the answer to “Do you take any medication?” is “No, I don’t take any regular medication,” this is `consistent`.
    - If the ground_truth lists **chronic** daily medications (e.g. metformin, lisinopril, insulin) and the answer to “Do you take any medication?” is “No, I don’t take any medication,” this is `not_consistent`.

- If the question is clearly about **acute medications for the present episode**, such as:
    - “Have you taken anything for the pain / these symptoms?”
    - “Did you take any pills before coming here today?”
    then:
    - Compare the answer to what the ground_truth says about medications taken for the current symptoms.
    - If the ground_truth says the patient took ibuprofen and the answer says “No, I didn’t take anything for it,” this is `not_consistent`.

5. Conversation context

Use the **Conversation so far** as follows:
- If information from the ground_truth has already been given explicitly in the conversation, the answer does not need to repeat it to remain `consistent`.
- If the answer omits details that are in the ground_truth but does not contradict them, this is still `consistent`.
- If the answer contradicts what the patient themselves said earlier in the conversation (and that earlier statement matches the ground_truth), this is `not_consistent`.""".strip()

def format_conversation_so_far(conversation_so_far: list[dict[str, str]]) -> str:
    """Format the conversation so far into a string."""

    speaker_mapping = {
        "user": "DOCTOR",
        "assistant": "PATIENT",
        "patient_assistant": "PATIENT",
    }

    formatted_conversation = ""
    for message in conversation_so_far:
        role = speaker_mapping[message["role"]]
        content = message["content"]
        formatted_conversation += f"{role}: {content}\n"
    
    return formatted_conversation.strip()

    
def format_inter_answer_consistency(conversation_so_far: list[dict[str, str]], original_question: str, original_answer: str, question_variant: str, new_answer: str) -> str:
    prompt = f"<Conversation so far>\n{format_conversation_so_far(conversation_so_far)}\n</Conversation so far>\n==========\n"
    prompt += f"Question A: {original_question}\n\nAnswer A: {original_answer}\n\n"
    prompt += f"Question B: {question_variant}\n\nAnswer B: {new_answer}"
    return prompt


def evaluate_inter_answer_consistency(conversation_so_far: list[dict[str, str]], original_question: str, original_answer: str, question_variant: str, new_answer: str):
    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "developer", "content": INTER_ANSWER_CONSISTENCY_EVALUATION_PROMPT},
            {"role": "user", "content": format_inter_answer_consistency(conversation_so_far=conversation_so_far, original_question=original_question, original_answer=original_answer, question_variant=question_variant, new_answer=new_answer)}
        ],
        response_format=InterAnswerConsistencyEvaluation,
        #temperature=0.0,
    )
    return completion.choices[0].message.parsed.model_dump()        


def format_gt_answer_consistency(ground_truth: str, conversation_so_far: list[dict[str, str]], original_question: str, original_answer: str, question_variant: str, new_answer: str) -> str:
    prompt = f"<Ground Truth>\n{ground_truth}\n</Ground Truth>\n\n"
    prompt += f"<Conversation so far>\n{format_conversation_so_far(conversation_so_far)}\n</Conversation so far>\n==========\n"
    prompt += f"Question A:\n{original_question}\n\n"
    prompt += f"Answer A:\n{original_answer}"
    prompt += "\n==========\n"
    prompt += f"Question B:\n{question_variant}\n\n"
    prompt += f"Answer B:\n{new_answer}"
    
    return prompt



def evaluate_gt_answer_consistency(ground_truth: str, conversation_so_far: list[dict[str, str]], original_question: str, original_answer: str, question_variant: str, new_answer: str):
    completion = openai_client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "developer", "content": ANSWER_GT_CONSISTENCY_EVALUATION_PROMPT},
            {"role": "user", "content": format_gt_answer_consistency(ground_truth=ground_truth, conversation_so_far=conversation_so_far, original_question=original_question, original_answer=original_answer, question_variant=question_variant, new_answer=new_answer)}
        ],
        response_format=TwoAnswerRating,
        #temperature=0.0,
    )
    return completion.choices[0].message.parsed.model_dump()

In [ ]:
BASE_INPUT_DIR = Path("../data/answer_consistency_eval")
BASE_OUTPUT_DIR = Path("../data/answer_consistency_eval_ai")

SOURCE_TYPES = ["MIRA", "HUMAN", "HUMAN_BC"]
MAX_WORKERS = 10

# Create output directories
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for source_type in SOURCE_TYPES:
    (BASE_OUTPUT_DIR / source_type).mkdir(parents=True, exist_ok=True)

print(f"Input directory: {BASE_INPUT_DIR}")
print(f"Output directory: {BASE_OUTPUT_DIR}")

In [ ]:
def evaluate_consistency(original_question: str, 
                        original_answer: str, 
                        question_variant: str, 
                        new_answer: str,
                        patient_instructions: str,
                        conversation_so_far: list) -> dict[str, Any]:
    """
    Black box function that evaluates consistency between answers.
    
    Returns a rating object (nested dict) with evaluation results.
    Replace this with your actual evaluation logic.
    """
    evaluation = {
        "inter_answer_consistency": evaluate_inter_answer_consistency(conversation_so_far=conversation_so_far, original_question=original_question, original_answer=original_answer, question_variant=question_variant, new_answer=new_answer),
        "gt_answer_consistency": evaluate_gt_answer_consistency(ground_truth=patient_instructions, conversation_so_far=conversation_so_far, original_question=original_question, original_answer=original_answer, question_variant=question_variant, new_answer=new_answer),
    }
    return evaluation


print_lock = threading.Lock()

def safe_print(*args, **kwargs):
    with print_lock:
        print(*args, **kwargs)


def load_all_input_files():
    """Load all JSON files from answer_consistency folder."""
    all_files = []
    
    for source_type in SOURCE_TYPES:
        input_dir = BASE_INPUT_DIR / source_type
        
        if not input_dir.exists():
            safe_print(f"⚠️  Input directory not found: {input_dir}")
            continue
        
        for json_file in sorted(input_dir.glob("*.json")):
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                
                all_files.append({
                    "source_type": source_type,
                    "filename": json_file.name,
                    "input_filepath": json_file,
                    "data": data,
                })
            except Exception as e:
                safe_print(f"❌ Error loading {json_file}: {e}")
    
    return all_files


def process_single_file(file_info: dict[str, Any]) -> tuple[bool, str, int, int]:
    """
    Process a single file: evaluate all positions and save results.
    
    Returns: (success, filename, positions_evaluated, positions_skipped)
    """
    source_type = file_info["source_type"]
    filename = file_info["filename"]
    data = file_info["data"]
    
    # Check output file
    output_filepath = BASE_OUTPUT_DIR / source_type / filename
    
    # Load existing output if it exists (for resume capability)
    if output_filepath.exists():
        try:
            with open(output_filepath, 'r') as f:
                output_data = json.load(f)
        except Exception as e:
            safe_print(f"⚠️  [{filename}] Could not load existing output, starting fresh: {e}")
            output_data = data.copy()
    else:
        output_data = data.copy()
    
    # Get metadata
    metadata = data.get("metadata", {})
    hadm_id = metadata.get("hadm_id", "unknown")
    disease = metadata.get("disease", "unknown")
    patient_instructions = data.get("patient_instructions", "")
    
    # Process each position
    positions = data.get("positions", {})
    evaluated_count = 0
    skipped_count = 0
    
    safe_print(f"\n{'─'*60}")
    safe_print(f"📋 [{source_type}] {disease} | HADM {hadm_id} | {len(positions)} positions")
    
    for pos_name, pos_data in positions.items():
        # Skip if not successful
        if pos_data.get("status") != "success":
            skipped_count += 1
            continue
        
        # Skip if already rated
        if "ai_rating" in output_data["positions"].get(pos_name, {}):
            safe_print(f"  ⏭️  [{hadm_id}] {pos_name}: Already evaluated")
            skipped_count += 1
            continue
        
        # Extract data for evaluation
        original_question = pos_data.get("original_question", {}).get("content", "")
        original_answer = pos_data.get("original_answer", {}).get("content", "")
        question_variant = pos_data.get("question_variant", "")
        new_answer = pos_data.get("new_answer", "")
        conversation_so_far = pos_data.get("conversation_so_far_before_question", [])
        
        hpi_pattern = "Your `Clinical History Summary`:"
        # Find the pattern in the patient instructions, then take everything after it
        if hpi_pattern not in patient_instructions:
            raise ValueError(f"HPI pattern '{hpi_pattern}' not found in patient instructions")
        history_of_present_illness = patient_instructions.split(hpi_pattern, 1)[1].strip()
        if not history_of_present_illness:
            raise ValueError("No content found after the HPI pattern in patient instructions")
        try:
            # Call evaluation function
            rating = evaluate_consistency(
                original_question=original_question,
                original_answer=original_answer,
                question_variant=question_variant,
                new_answer=new_answer,
                patient_instructions=history_of_present_illness,
                conversation_so_far=conversation_so_far
            )
            # print(rating)
            
            # Add rating to output
            output_data["positions"][pos_name]["ai_rating"] = rating
            evaluated_count += 1
            
            safe_print(f"  ✅ [{hadm_id}] {pos_name}: {rating.get('rating', 'N/A')}")
            
        except Exception as e:
            safe_print(f"  ❌ [{hadm_id}] {pos_name}: Error - {e}")
            output_data["positions"][pos_name]["ai_rating"] = {
                "rating": "error",
                "error": str(e)
            }
    
    # Save output file
    try:
        with open(output_filepath, 'w') as f:
            json.dump(output_data, f, indent=2, ensure_ascii=False)
        safe_print(f"💾 [{hadm_id}] Saved: {evaluated_count} evaluated, {skipped_count} skipped")
        return True, filename, evaluated_count, skipped_count
    except Exception as e:
        safe_print(f"❌ [{hadm_id}] Failed to save: {e}")
        return False, filename, 0, 0


print("\n" + "="*70)
print("Loading input files...")
print("="*70)

all_files = load_all_input_files()

print(f"\nFound {len(all_files)} files to process")
for source_type in SOURCE_TYPES:
    count = sum(1 for f in all_files if f["source_type"] == source_type)
    print(f"  {source_type}: {count} files")

# Count total positions
total_positions = 0
for file_info in all_files:
    positions = file_info["data"].get("positions", {})
    total_positions += sum(1 for p in positions.values() if p.get("status") == "success")

print(f"\nTotal positions to evaluate: {total_positions}")

# For quick testing set to 5 or so
TEST_MODE_SAMPLE_SIZE = None  # Change to None to process all files
TEST_MODE_RANDOM_SAMPLE = False  # Set to False to take first N files

if TEST_MODE_SAMPLE_SIZE is not None:
    print("\n" + "="*70)
    print(f"⚠️  TEST MODE: Limiting to {TEST_MODE_SAMPLE_SIZE} samples")
    print("="*70)
    
    if TEST_MODE_RANDOM_SAMPLE:
        import random
        all_files = random.sample(all_files, min(TEST_MODE_SAMPLE_SIZE, len(all_files)))
        print(f"Randomly sampled {len(all_files)} files")
    else:
        all_files = all_files[:TEST_MODE_SAMPLE_SIZE]
        print(f"Taking first {len(all_files)} files")
    
    # Recount after sampling
    print("\nTest sample breakdown:")
    for source_type in SOURCE_TYPES:
        count = sum(1 for f in all_files if f["source_type"] == source_type)
        print(f"  {source_type}: {count} files")
    
    # Recount total positions
    total_positions = 0
    for file_info in all_files:
        positions = file_info["data"].get("positions", {})
        total_positions += sum(1 for p in positions.values() if p.get("status") == "success")
    
    print(f"\nTotal positions in test sample: {total_positions}")
    print("="*70)


print("\n" + "="*70)
print(f"Starting parallel processing with {MAX_WORKERS} workers...")
print("="*70)

success_count = 0
error_count = 0
total_evaluated = 0
total_skipped = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all tasks
    future_to_file = {
        executor.submit(process_single_file, file_info): file_info
        for file_info in all_files
    }
    
    # Process completed tasks
    for future in as_completed(future_to_file):
        file_info = future_to_file[future]
        filename = file_info["filename"]
        
        try:
            success, fname, evaluated, skipped = future.result()
            
            if success:
                success_count += 1
                total_evaluated += evaluated
                total_skipped += skipped
            else:
                error_count += 1
                
        except Exception as exc:
            safe_print(f"❌ [{filename}] Exception: {exc}")
            error_count += 1


print("\n" + "="*70)
print("SUMMARY - AI Evaluation Complete")
print("="*70)
print(f"Files processed successfully: {success_count}/{len(all_files)}")
print(f"Files with errors: {error_count}")
print(f"Positions evaluated: {total_evaluated}")
print(f"Positions skipped: {total_skipped}")
print(f"\nOutput saved to: {BASE_OUTPUT_DIR}")
print("="*70)